# CSE 25 – Introduction to Artificial Intelligence  
## Worksheet 15: Q-Learning in Practice

**Today’s focus:**  
How do we implement and train a Q-learning agent to play a real game, and what does the learning process look like in detail?

### Guiding Questions
1. How do we represent and update the Q-table in code?
2. What is the role of the temporal difference and the learning rate in Q-learning?
3. How does the agent balance exploration and exploitation in practice?
4. How do we evaluate the performance of a trained agent?
5. What challenges arise when training an agent in a real environment?

### Learning Objectives
By the end of this worksheet, you will be able to:
- Implement the Q-learning update rule in Python.
- Apply the $\epsilon$-greedy strategy for action selection in code.
- Train an agent to play misère Nim using Q-learning.
- Analyze the learning curve and performance of your agent.
- Extract and interpret the optimal policy from a learned Q-table.

**Instructions:**  
Create a copy of this notebook and complete it during class.  
Work through the cells below **in order**.

You may discuss with your neighbors, but make sure you understand  
what each step is doing and why.

**Submission**  
When finished, download the notebook as a PDF and upload it to Gradescope under `In-Class – Week 9 Thursday`.

To download as a PDF on DataHub:  
`File -> Save and Export Notebook As -> PDF`

<img src="images/agent-env-mdp.png" width="500">

#### The Agent–Environment Interaction

Reinforcement learning is built around repeated interaction between the **agent** and the **environment**.

At each step:

1. The agent observes the current **state** $s$ of the environment.
2. The agent chooses an **action** $a$ from allowed actions.
3. The environment 
   - responds with a **reward** $r$
   - moves to the **next state** $s'$


#### Key Components

- **State ($s$)**: A representation of the environment.
- **Action ($a$)**: A choice the agent can make in that state to affect the environment.
- **Reward ($r$)**: A numerical signal from the environment indicating how good (or bad) the outcome was.
- **Next state ($s'$)**: The updated representation of the environment that results after taking action $a$ in state $s$.

### Markov Decision Process (MDP)

An MDP is a mathematical model for sequential decision-making when outcomes are *uncertain*.

It consists of:

- A set of **states** $S$ called the *state space*.
- A set of **actions** $A$ called the *action space*. 
- **Transition probabilities** $P(s' \mid s, a)$ which describe how likely we are to move to state $s'$ after taking action $a$ in state $s$. 
- A **reward function** $R(s, s', a)$ which gives the immediate reward received when the agent takes action $a$ in state $s$ and transitions to state $s'$. Rewards are **scalar values**. Higher values usually indicate better outcomes.

An MDP is a tuple of ($S$, $A$, $P(s' \mid s, a)$, $R(s, s', a)$)

#### Policy

A **policy** $\pi$ is a function that maps the state space $S$ to the action space $A$. It may be deterministic or stochastic.

#### Optimization Objective
**The goal** of reinforcement learning is to *learn a policy* that maximizes **total expected reward over time**.

### Tabular Q-Learning 

In Q-learning, we store Q-values denoted as $Q(s, a)$ in a table. This value represents the **total expected reward** for taking action $a$ in state $s$ and then continuing to make the **best possible decisions** until the task is complete.

It essentially converts a short-term move into a long-term "quality" score, where higher values indicate a more successful path to the goal.

| State $s$| Take 1 | Take 2 |
|---|---|---|
| 1 | $Q(1,1)$ | $Q(1,2)$ |
| 2 | $Q(2,1)$ | $Q(2,2)$ |
| 3 | $Q(3,1)$ | $Q(3,2)$ |
| 4 | $Q(4,1)$ | $Q(4,2)$ |
| 5 | $Q(5,1)$ | $Q(5,2)$ |
| 6 | $Q(6,1)$ | $Q(6,2)$ |

In [ ]:
# Q-table as a flat dictionary with tuple keys: (state, action)
# States: 0..6, Actions: 1 or 2

Q = {(state, action): 0.0 for state in range(1, 7) for action in (1, 2)}

def display_q_table(Q):
    states = sorted({state for state, _ in Q})

    print(f"{'State':>5} {'Take 1':>10} {'Take 2':>10}")
    print("-" * 27)

    for state in states:
        q_take_1 = Q.get((state, 1), float("nan"))
        q_take_2 = Q.get((state, 2), float("nan"))
        print(f"{state:>5} {q_take_1:>10.3f} {q_take_2:>10.3f}")

display_q_table(Q)
print(Q)

#### Temporal Difference Learning


$$\mu_t = \mu_{t-1} + \alpha_t(x_t - \mu_{t-1})$$

The corrective term $x_t-\mu_{t-1}$ is known as a *temporal difference*. It represents the "error"—the difference between the new sample and our current guess.


We can use this temporal difference update to update our Q-values as we interact with the environment and get rewards. 

We do not want to completely replace $Q(s,a)$. Instead, we move it slightly toward the new estimate:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha ( x_t - Q(s,a) )
$$

where $\alpha$ is the learning rate.


Now the question is: **What is $x_t$, also known as the TD-target, for Q-learning?**

In Q-Learning, our "sample" $x_t$ must account for both immediate rewards and future potential. 

After taking action $a$ in state $s$, we observe:
- An immediate reward **$r$**
- The next state **$s'$**

An action might not give a reward immediately, but it might lead to a good(or a bad) position. So, our target $x_t$ is the **reward now plus how good the next state looks**:

$$x_t = r + \max_{a'} Q(s',a')$$

**What does the $\max_{a'} Q(s',a')$ term mean?**

Think of this as the agent **looking one step ahead**. Once the agent lands in the next state ($s'$), it checks its Q-table for all possible actions it could take from there. By picking the **maximum** ($\max$), it is asking: *"What is the highest-rated move I can make from this new position?"* We are essentially saying that the "quality" of our current move depends on the best possible move we can make next.

This is called **Bootstrapping**. We update our estimate for $Q(s, a)$ using our **current best estimate** of the next state's Q-values. We are literally updating an estimate with another estimate! As the agent experiences actual rewards, whether they happen immediately or further down the line, our **estimates get better**. These improvements gradually "propagate back" to make the estimates of all the previous moves better in our Q-table.

**Adding the Final Piece: The Discount Factor ($\gamma$)**

If we simply added the future values directly, our Q-values could grow infinitely large. We also want to give our agent a sense of **urgency** - a reward of $+1$ right now should be worth more than a reward of $+1$ ten moves from now.

To handle this, we introduce the **Discount Factor ($\gamma$)**, a value between $0$ and $1$. 

Our final, polished target ($x_t$) becomes:
$$x_t = r + \gamma \max_{a'} Q(s',a')$$

#### **The Full Q-Update Equation**

Plugging our discounted target back into the update formula, we get the complete rule for training our AI:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big( \underbrace{[r + \gamma \max_{a'} Q(s',a')]}_{\text{New Evidence}} - \underbrace{Q(s,a)}_{\text{Old Guess}} \Big)$$

#### The Decision Dilemma: Exploration vs. Exploitation

Now that we have a mathematical way to update our "memory" ($Q$-values), how does the agent actually pick a move during the game? 

It faces a classic problem called the **Exploration vs. Exploitation Dilemma**:

* **Exploitation**: Choosing the move that currently has the highest score in our table. (Like going to your favorite restaurant because you *know* it's good).
* **Exploration**: Trying a random move to see if it leads to an even better outcome. (Like trying a brand new restaurant you've never been to).



If our agent only ever "exploits" early on, it might get stuck in a mediocre strategy just because it worked once. To find the truly optimal way to play, it must be curious!

#### The $\epsilon$-greedy Strategy

To balance these two needs, we use the **$\epsilon$-greedy strategy**. We define a parameter $\epsilon \in [0, 1]$ which determines the probability of exploration.

**The Decision Logic:**
1. Generate a random number (probability) $p \in [0, 1]$.
2. **If** $p < \epsilon$:
    - **Explore**: Select an action $a$ uniformly at random from the set of all legal actions.
3. **Else**:
    - **Exploit**: Select the action $a$ that maximizes the current Q-value: $a = \text{argmax}_{a'} Q(s, a')$.

In [ ]:
import random

# To generate random numbers between 0 and 1 we can use the random.random().
print(random.random())

# We can use this to implement an epsilon-greedy policy for action selection.

In [ ]:
epsilon = 0.1
exploration = 0
exploitation = 0

# If random.random() < epsilon, we choose a random action (exploration), 
# otherwise we choose the action with the highest Q-value (exploitation).
for _ in range(5000):
    if random.random() < epsilon:
        exploration += 1
    else:
        exploitation += 1

print(f"Exploration count: {exploration}, Exploitation count: {exploitation}")
print(f"Exploration percentage: {exploration / 5000 * 100:.2f}%, Exploitation percentage: {exploitation / 5000 * 100:.2f}%")

Q. What happens if we set $\epsilon$ to 1?

`YOUR ANSWER HERE`

Q. What happens if we set $\epsilon$ to 0?

`YOUR ANSWER HERE`

### The Q-Learning Algorithm

Now that we have the update equation and a strategy for selecting actions, we can combine them into a complete learning loop. The goal of this algorithm is to iteratively improve our Q-table until the values accurately represent the quality of every possible move.

**Algorithm Logic:**

1.  **Initialize**: Set all $Q(s, a)$ values to 0 (the agent starts with no knowledge).
2.  **Repeat for each episode**:
    * Set the starting state $s$ (e.g., $s = 6$ in Nim).
    * **For each step in the episode**:
        * **Select** an action $a$ to apply in state $s$ using the **$\epsilon$-greedy** strategy.
        * **Execute** action $a$ and observe the outcome: the **reward** $r$ and the **next state** $s'$.
        * **Calculate the TD Error ($\delta$)**: 
            $$\delta \leftarrow r + \gamma \max_{a'} Q(s',a') - Q(s,a)$$
        * **Update the Table**: Apply the nudge to the memory:
            $$Q(s,a) \leftarrow Q(s,a) + \alpha \cdot \delta$$
        * **Transition**: Update our current position to the next state: $s \leftarrow s'$.
    * **Stop** when $s$ is a terminal state (the game is over).
3.  **End** training once the Q-values converge (they stop changing significantly).

#### Policy Extraction

After the training loop finishes, we have an "optimal" Q-table. However, a table of numbers isn't a strategy yet. We need to perform **Policy Extraction** to get our final rules for playing the game perfectly.

A **Policy ($\pi$)** is a mapping from states to actions. We extract the optimal policy by simply choosing the action that has the highest Q-value for every state:

$$\pi(s) = \text{argmax}_{a} Q(s,a)$$

In [ ]:
# Q-table as a flat dictionary with tuple keys: (state, action)
# States: 0..6, Actions: 1 or 2
random.seed(42)  # For reproducibility
# Initializing Q-table with random values for demonstration
Q = {(state, action): random.random() for state in range(1, 7) for action in (1, 2)}

display_q_table(Q)

In [ ]:
# Optimal Policy based on Q-table - argmax action for each state
optimal_policy = {}
for state in range(1,7):
    q_take_1 = Q[(state, 1)]
    q_take_2 = Q[(state, 2)]
    optimal_policy[state] = 1 if q_take_1 > q_take_2 else 2

print("Optimal Policy (state: best action):")
for state, action in optimal_policy.items():
    print(f"State {state}: Action {action}")

### EXERCISE: Training agent to play misère Nim

Nim is a turn-based game where players remove objects from a pile, and depending on the version, the player who takes the last object either wins (standard Nim) or loses (misère Nim).

#### The Game Rules (Our Version)
- We start with one pile with **6 objects**.
- On each turn, a player may remove **1 or 2 objects**.
- The player who takes the last object **loses**(misère Nim).

#### States, Actions and Rewards for our version of Nim

We have 6 states: 1, 2, 3, 4, 5, 6
Each number represents a *state* (the number of objects remaining).

We have 2 actions: 
- "1" (remove 1 object)
- "2" (remove 2 objects)

**Reward ($r$)**:
  - $+1$ for a win,
  - $-1$ for a loss,
  - $0$ at all intermediate steps.

The agent will play first in our version, similar to what we did in the in-class demo.

In [ ]:
# Some helper functions for misère NIM Game 

random.seed(2)  # For reproducibility

def get_legal_actions(state):
    '''Returns a list of legal actions based on the current state'''
    # If there are 2 or more objects, you can take either 1 or 2. 
    # If there's only 1 object left, you can only take 1.
    return [1, 2] if state >= 2 else [1]

def apply_action(state, action):
    '''Returns the next state after applying the action to the state'''
    # We assume the action is legal
    return state - action

def get_q_value(Q, state, action):
    '''Returns the Q-value for a given state-action pair from the Q-table'''
    return Q[(state, action)] # We could also use Q.get((state, action), 0.0) to return 0.0 for unseen pairs - this avoids KeyError but we are initializing all pairs, so it's not strictly necessary.

def get_max_q_value(Q, state):
    '''Returns the best future Q-value for a given a state'''
    
    # If the state is 0 or negative, the game is over and there are no future Q-values.
    if state <= 0:
        return 0.0
    
    # Otherwise, we look at all legal actions from the next state and return the maximum Q-value among them.
    legal_next_actions = get_legal_actions(state)
    
    # Initialize a list to store the Q-values for the legal actions
    q_values = []
    for action in legal_next_actions:
        q_values.append(get_q_value(Q, state, action))  # This is just to illustrate that we are accessing the Q-values for the legal actions.
    
    # Return the maximum Q-value
    return max(q_values)

#### Q. Complete the select_action and update_q_table functions below.

In [ ]:
def select_action(Q, state, epsilon):
    ''' Returns an action using epsilon-greedy policy based on the current Q-table and the given state'''
    
    # First, we get the legal actions for the given state.
    legal_actions = get_legal_actions(state)
    p = random.random()
    
    # To select a random action from a list of legal actions, we can use random.choice(legal_actions).
    return random.choice(legal_actions) # TODO: Comment/Remove this line after implementing the epsilon-greedy policy
    # YOUR CODE HERE
    # With probability epsilon, we will explore by choosing a random action from the legal actions.
    # if p < epsilon: 
        # EXPLORE
    # else:
        # EXPLOIT
        # With probability (1 - epsilon), we will exploit by choosing the action with the highest Q-value for the current state.

        # 1. Get the max Q-value for the current state
        # 2. Find which actions have that value
        # 3. Return a choice from those actions
        # If there are multiple actions with the same max Q-value, we will randomly choose among them to break ties.

    
def update_q_table(Q, state, action, reward, next_state, alpha, gamma):
    ''' Updates the Q-table based on the observed transition (state, action, reward, next_state) 
        using the learning rate (alpha) and discount factor (gamma)
        The update is based on the Q-learning formula:
            delta = reward + gamma * max(Q(next_state)) - Q(state, action)
            Q(state, action) = Q(state, action) + alpha * delta'''
    
    # YOUR CODE HERE
    # STEP 1: Get current Q-value and the max Q-value of the next state
    # current_q = ...
    # next_max = ...
    
    # STEP 2: Calculate the TD Error (delta)
    # formula: reward + gamma * next_max - current_q
    # delta = ...
    
    # STEP 3: Update the Table (Apply the nudge)
    # Q[(state, action)] = ... 


In [ ]:
def test_agent(Q, num_games=100):
    wins = 0
    losses = 0

    for _ in range(num_games):
        state = 6 # Start with 6 objects
        
        while state > 0:
            # 1. AGENT'S TURN
            # We set epsilon to 0 because we want the agent to use 
            # its learned knowledge, not explore randomly!
            action = select_action(Q, state, epsilon=0)
            state = apply_action(state, action)

            if state == 0: # Agent took the last object -> Loss
                losses += 1
                break

            # 2. OPPONENT'S TURN 
            # Random opponent: chooses a random legal action
            opp_action = random.choice(get_legal_actions(state))
            state = apply_action(state, opp_action)

            if state == 0: # Opponent took the last object -> Agent Wins
                wins += 1
                break
                
    win_rate = (wins / num_games) * 100
    return win_rate

#### Q. Update the reward values for win/loss:

In [ ]:
# STEP 1: Initialize Q-table with 0.0 for all state-action pairs
Q = {(state, action): 0.0 for state in range(1, 7) for action in (1, 2)}

# Set Hyperparameters
num_episodes = 200
alpha = 0.1  # Learning rate
gamma = 0.9  # Discount factor
epsilon = 0.1  # Exploration rate

# History of win rates for plotting
episode_numbers = []
win_rates = []

# Training loop
for i in range(1, num_episodes + 1):
    # New Game: Reset the environment to the initial state
    state = 6  # Set the starting game state
    

    while state > 0: # The game continues until we reach a terminal state (0 or negative)
        # STEP 2: AGENT  selects an action using epsilon-greedy
        action = select_action(Q, state, epsilon)
        
        # STEP 3: We execute action and observe outcome
        # We use 'temp_state' because the opponent needs to move before 
        # the agent sees the true 'next_state'.
        temp_state = apply_action(state, action)
        reward = 0  # Initialize reward

        if temp_state == 0:
            # If the agent took the last object what shuold be the reward?
            
            reward = 0 # YOUR CODE HERE - Change this to the correct reward
            next_state = temp_state
        else:
            # OPPONENT'S TURN
            opp_action = random.choice(get_legal_actions(temp_state)) # Opponent takes a random legal action
            next_state = apply_action(temp_state, opp_action)
            
            if next_state == 0:
                # If the opponent took the last object, what should be the reward?
                reward = 0 # YOUR CODE HERE - Change this to the correct reward
                pass
            else:
                # Game continues
                reward = 0
        
        # STEP 4: Update the Q-Table
        update_q_table(Q, state, action, reward, next_state, alpha, gamma)
        
        # STEP 5: Transition (Update current position to next state)
        state = next_state
    
    if i==1 or i % 10 == 0:  # Display Q-table every 10 episodes
        win_rates.append(test_agent(Q, 100))  # Test the agent and store the win rate for plotting
        episode_numbers.append(i)
        print(f"\nEpisode {i}")
        display_q_table(Q)
        
        

print("Training complete!")

Now let's test our agent using the extracted policy from the learned Q-table:

In [ ]:
# Let's test the untrained agent first to see how it performs before training.
Q_untrained = {(state, action): 0.0 for state in range(1, 7) for action in (1, 2)}
print("\nTesting the untrained agent...")
print(test_agent(Q_untrained, 100))

# Now let's test the trained agent to see how much it has improved!
print("\nTesting the trained agent...")
print(test_agent(Q, 100))


In [ ]:
# Plotting the win rates over time
import matplotlib.pyplot as plt
plt.plot(episode_numbers, win_rates)
plt.xlabel('Episode')
plt.ylabel('Win Rate (%)')
plt.title('Agent Win Rate Over Time')
plt.grid()
plt.show()